# MMDS Phase 0 CPU verification

Attach `saisathwikregula/mmds-dataset-23013` as an input and select CPU. This notebook checks one file path without opening a dataset file. The Spark operation uses synthetic data. Internet is required only if the pinned PySpark package must be installed.

In [ ]:
from pathlib import Path
import platform
import subprocess
import sys

print(f"Python {platform.python_version()}", flush=True)
if sys.version_info < (3, 11):
    raise RuntimeError("PySpark 4.3.0 requires Python 3.11 or newer.")

sample = Path("Cyberattack-MMDS/dataset/raw/Friday-02-03-2018_TrafficForML_CICFlowMeter.csv")
roots = (
    Path("/kaggle/input/datasets/saisathwikregula/mmds-dataset-23013"),
    Path("/kaggle/input/mmds-dataset-23013"),
)
mounted_file = next((root / sample for root in roots if (root / sample).is_file()), None)
if mounted_file is None:
    raise FileNotFoundError("Attach saisathwikregula/mmds-dataset-23013, then rerun.")
print(f"Dataset mount verified: {mounted_file}", flush=True)
subprocess.run(["java", "-version"], check=True)

In [ ]:
import os
from importlib.metadata import PackageNotFoundError, version

expected_spark = "4.3.0"
try:
    installed_spark = version("pyspark")
except PackageNotFoundError:
    installed_spark = None

if installed_spark != expected_spark:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", f"pyspark=={expected_spark}"],
        check=True,
    )

os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
os.environ["PYSPARK_PYTHON"] = sys.executable
from pyspark import __version__ as pyspark_version
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master("local[2]")
    .appName("mmds-phase-0-kaggle")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
try:
    if spark.version != expected_spark or pyspark_version != expected_spark:
        raise RuntimeError(f"Unexpected versions: Spark {spark.version}, PySpark {pyspark_version}")
    if spark.range(4).count() != 4:
        raise RuntimeError("Unexpected result from the synthetic Spark job.")
    print(f"PASS: Spark {spark.version}, PySpark {pyspark_version}", flush=True)
    Path("/kaggle/working/phase0_status.txt").write_text(
        f"PASS: Python {platform.python_version()}, Spark {spark.version}, "
        f"PySpark {pyspark_version}; dataset mount verified.\n",
        encoding="utf-8",
    )
finally:
    spark.stop()